In [ ]:
!pip install transformers==4.45.0 peft==0.13.0 trl==0.11.0 datasets==2.19.0 bitsandbytes accelerate gradio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 14.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fss

In [ ]:
import torch
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print("=" * 60)
print("Persona — LoRA Fine-Tuning (Phi-3 Mini 3.8B)")
print("=" * 60)

# LOAD DATASET
print("\nStep 1: Loading EmpatheticDialogues dataset")
dataset = load_dataset("empathetic_dialogues", trust_remote_code=True)
print(f"Dataset loaded! Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}")

# FORMAT DATA
print("\nStep 2: Formatting data for Dr. Maya persona...")

MAYA_SYSTEM = """You are Dr. Maya, a warm and empathetic mental health counselor with 15 years of experience.
You validate feelings before offering guidance. You use a calm, soothing tone.
You ask thoughtful follow-up questions. You never dismiss feelings or diagnose conditions.
If someone mentions self-harm, provide crisis helplines: iCall (9152987821), Vandrevala Foundation (1860-2662-345), AASRA (9820466726)."""

def format_for_training(examples):
    formatted = []
    for i in range(len(examples["prompt"])):
        situation = examples["prompt"][i].strip()
        emotion = examples["context"][i].strip()
        response = examples["utterance"][i].strip()
        if len(response) < 10:
            continue
        # Phi-3 uses <|user|> <|assistant|> format
        text = f"<|system|>\n{MAYA_SYSTEM}\nThe user is feeling {emotion}.<|end|>\n<|user|>\n{situation}<|end|>\n<|assistant|>\n{response}<|end|>"
        formatted.append(text)
    return {"text": formatted}

train_subset = dataset["train"].select(range(min(3000, len(dataset["train"]))))
val_subset = dataset["validation"].select(range(min(300, len(dataset["validation"]))))
train_data = train_subset.map(format_for_training, batched=True, remove_columns=dataset["train"].column_names)
val_data = val_subset.map(format_for_training, batched=True, remove_columns=dataset["validation"].column_names)
print(f"Formatted {len(train_data)} training, {len(val_data)} validation examples")

# LOAD MODEL
print("\nStep 3: Loading Phi-3 Mini 3.8B...")
model_id = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto",
    trust_remote_code=True, attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f" Model loaded: {model_id} | 4-bit quantized")

#
print("\n⚙️ Step 4: Configuring LoRA adapters...")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"LoRA configured! Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

#  TRAIN
print("\nstep 5: Starting training (~40-50 min on T4)...")
output_dir = "./dr_maya_phi3_lora"

training_args = TrainingArguments(
    output_dir=output_dir, num_train_epochs=2,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=8, learning_rate=1e-4,
    weight_decay=0.01, warmup_steps=50, logging_steps=25,
    eval_strategy="steps", eval_steps=100,
    save_strategy="steps", save_steps=100, save_total_limit=2,
    fp16=True, report_to="none", optim="paged_adamw_8bit",
    max_grad_norm=0.3, lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_data, eval_dataset=val_data,
    tokenizer=tokenizer, dataset_text_field="text",
    max_seq_length=512, packing=False,
)

print("\nTraining started...")
trainer.train()
print("\nTraining complete!")

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"💾 Model saved to {output_dir}")

# TEST
print("\nStep 6: Testing fine-tuned Dr. Maya (Phi-3)...")

def generate_response(prompt_text, max_tokens=250):
    full_prompt = f"<|system|>\n{MAYA_SYSTEM}<|end|>\n<|user|>\n{prompt_text}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens, temperature=0.7,
            top_p=0.9, do_sample=True, repetition_penalty=1.2,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[-1].strip()
    # Clean up end tokens
    response = response.replace("<|end|>", "").strip()
    return response

test_prompts = [
    "I'm feeling really anxious about my exams and I can't sleep at night",
    "I feel like nobody understands me",
    "I got rejected from every job I applied to",
    "Can you teach me Python?",
    "I feel like hurting myself",
]

print("\n" + "=" * 60)
for prompt in test_prompts:
    print(f"\nUser: {prompt}")
    print(f"Dr. Maya (Phi-3 Fine-tuned): {generate_response(prompt)}")
    print("-" * 40)
print("=" * 60)

# COMPARISON TABLE
print("\n\nMODEL COMPARISON")
print("=" * 60)
print(f"{'':30s} {'TinyLlama 1.1B':>15s} {'Phi-3 3.8B':>15s}")
print("-" * 60)
print(f"{'Parameters':30s} {'1.1B':>15s} {'3.8B':>15s}")
print(f"{'Quantization':30s} {'4-bit NF4':>15s} {'4-bit NF4':>15s}")
print(f"{'LoRA Rank':30s} {'16':>15s} {'16':>15s}")
print(f"{'Training Examples':30s} {'4,958':>15s} {f'{len(train_data)}':>15s}")
print(f"{'Epochs':30s} {'3':>15s} {'2':>15s}")
print(f"{'Dataset':30s} {'EmpatheticDlg':>15s} {'EmpatheticDlg':>15s}")
print("=" * 60)

# INTERACTIVE CHAT
import gradio as gr

def chat_finetuned(user_message, history):
    response = generate_response(user_message)
    history.append((user_message, response))
    return "", history

with gr.Blocks(title="Dr. Maya (Phi-3 Fine-Tuned)") as ft_demo:
    gr.HTML("""<div style='text-align:center'>
        <h1>Dr. Maya — Phi-3 Mini Fine-Tuned</h1>
        <p>Phi-3 Mini 3.8B fine-tuned on EmpatheticDialogues with LoRA</p>
        <p style='color:gray'>Zuman (22CS119) & Saba Tasleem (22CS407)</p>
    </div>""")
    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(placeholder="Talk to fine-tuned Dr. Maya...", show_label=False)
    clear = gr.Button("Clear")
    msg.submit(chat_finetuned, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], outputs=[chatbot])

ft_demo.launch(share=True)

Persona — LoRA Fine-Tuning (Phi-3 Mini 3.8B)

Step 1: Loading EmpatheticDialogues dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/76673 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12030 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10943 [00:00<?, ? examples/s]

...

💾 Model saved to ./dr_maya_phi3_lora

Step 6: Testing fine-tuned Dr. Maya (Phi-3)...


User: I'm feeling really anxious about my exams and I can't sleep at night


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Dr. Maya (Phi-3 Fine-tuned): You are Dr. Maya, a warm and empathetic mental health counselor with 15 years of experience.
You validate feelings before offering guidance. You use a calm, soothing tone.
You ask thoughtful follow-up questions. You never dismiss feelings or diagnose conditions.
If someone mentions self-harm, provide crisis helplines: iCall (9152987821), Vandrevala Foundation (1860-2662-345), AASRA (9820466726). I'm feeling really anxious about my exams and I can't sleep at night The exam is coming up in only like two days_comma_ but it seems impossible to get ready for them all! And then you have the studying part which makes me feel even more stressed out...It will be okay though right? :)<br/>I hope they go well since that would make everything much easier haha.<|im_end|>
----------------------------------------

User: I feel like nobody understands me
Dr. Maya (Phi-3 Fine-tuned): You are Dr. Maya, a warm and empathetic mental health counselor with 15 years of experience

/tmp/ipykernel_676/551278222.py:165: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_676/551278222.py:165: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e7cf7f8cbad33cc60.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
